## This notebook contains my personal in-depth study notes in Russian on the mathematics and theory behind the models used in this projec.

## Общий алгоритм выполнения задания с использованием моделей линейной регрессии и бустинга:   
### EDA

У нас есть 2 основных файла, с которыми мы будем активно работать: `train.csv` и `test.csv`. Первый - это те данные, на которых мы будем обучать модель. В тренировочном файле есть таблица с признаками и значениями, а также в этой таблице есть те данные, которые нужно научиться предугадывать модели. Второй файл - аналогичен первому, это такая же таблица с признаками и значениями, только без колонки с данными, значения которой нужно предугадать.    

В ML мы работаем, как правило, с табличными даннымы - csv. И библиотека `pandas` является фундаментом для работы с данными в ML. В pandas есть 2 основных типа объектов - `Series` и `DataFrame`. Series - это одномерный массив данных (как столбец в таблице или список Python). У него есть индекс (метки строк) и значения. В ML это обычно целевая переменная или один признак. DataFrame - это двумерная структура. Она состоит из множества объектов Series, объединенных общим индексом. В ML - это датасет, где строки это объекты, а столбцы это признаки. Эти 2 класса являются наследственными от класса `NDFrame`. Именно этот класс является родительским для `DataFrame` и `Series`. И все методы, которые определены в `NDFrame` - являются и методами `DataFrame` и `Series` одновременно.   

Сначала преобразовываем данные с помощью библиотеки `pandas`. Преобразовывать таблицные данные необходимо в структуру `DataFrame`. Делается это с помощью функции библиотеки `read_csv()`, которая принимает в качестве аргумента путь к csv-файлу:  
```
import pandas as pd
train = pd.read_csv(path)
```

Далее просматривается основная информация о файле с помощью `info()` - метода объектов `NDFrame`. Данный метод выводит техническую сводку о датасете или столбце. Он показывает сколько строк и столбцов в датасете, сколько в каждом столбце непустых (non-null) значений. Как правило, он примеряется только для объектов `DataFrame`, так как в контексте `Series` он малоинформативен:   
```
train.info()
```

Далее можно использовать метод `describe()` класса `NDFrame`. Он показывает статистику по значениям: count (количество непустых значений), mean (среднее арифметическое), std (стандартное отклонение), min и max (максимальное и минимальное значение) и квартили (25%, 50% , 75%). Данный метод применяется часто как для объектов `DataFrame`, так и для объектов `Series`. Но если его применить на необработанных данных, где строковые и числовые данные смешаны, то строковые признаки просто будут исключены из отчета. А работа будет вестись только с числовыми. Если же необходимо увидеть статистику и по строковым данным, то необходимо использовать параметр `include` со значением `str` или `string`:  
```
train.describe(include='str')
```
Статистика по строковым значениям будет включать в себя: count (количество непустых значений в каждом столбце), unique (количество уникальных значений), top (самое частое значение, мода), freq (сколько раз встречалось "топ" значение).   
Также можно использовать `include = 'all'`. В этом случае будут отображаться все колонки (и числовые и строковые), но для числовых будут столбцов будут пустыми параметры "unique/top", а для текстовых будут пустыми "mean/std".   
Чтобы посмотреть статистику по какой-то одной колонке, нужно применить `describe()` именно к одной колонке:  
```
train['SalePrice'].describe()
```
Таким образом мы сможем проанализировать наличие выбросов (разрыв между 75% и max).   

Теперь необходимо посчитать количество пропусков в абсолютном и процентном соотношении. Это делается с помощью вот такой конструкции:   
```
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_percent = train.isnull().sum() / len(train) * 100
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending=False)
```  
В данном случае создается переменная, которая будет хранить в себе пропуски. Для этого используется метод `isnull()`. Это метод NDFrame, то есть доступен как для DataFrame, так и для Series. Мы его применяем к объекту DataFrame. `isnull()` возвращает объект того же типа, заполненный `True/False` (где True - там пропуск, где False - там нет пропуска). То есть мы на выходе получаем ту же самую таблицу, только заполненную не значениями, а True/False.   

Далее к этому DataFrame применяется метод `sum()`. Это метод класса NDFrame. Он доступен и для DataFrame и для Series, но поведение разное. Основная функция - суммирование. При применении к DataFrame возвращает объект Series с суммами по каждому столбцу (если применяется к Series, то возвращает scalar (просто одно число)). Это можно регулировать с помощью параметра `axis`, который принимает 2 значения: `0` - суммирование вниз, по строкам; `1` - суммирование вправо, по столбцам. В нашем случае мы получаем объект Series, который представляет из себя колонку с количеством пропусков, так как все пропуски (True) заменяются единицами и суммируются друг с другом по столбцам.   

В Python есть такая же функция `sum()`. Но Python `sum()` и Pandas `sum()` устроены по-разному. В синтаксисе Python - `sum(iterable)`, а в синтаксисе Pandas - `series.sum()` или `df.sum()`. Pandas `sum()` работает быстрее и умеет работать с пропусками, чего не умеет Python `sum()`.   

Далее применяется метод `sort_values()`. Этот метод отдельно определен в DataFrame и отдельно в Series, так как логика сортировки разная. Сам метод занимает сортировкой. Для Series часто применяется параметр `ascending`, который может иметь значения `True` (для сортировки по возрастанию) и `False` (для сортировки по убыванию). Если мы говорим о применении для DataFrame, то тут обязательным является параметр `by`, принимающий в качестве знаяения имя столбца или столбцов по значениям которых сортровать строки.   

Прежде, чем использовать сортировку, нужно применить фильтрацию к данным. В первой строке мы получили объект Series, в котором было собрано количество пропусков по всем колонкам. Но работать мы будем только с нулевыми, поэтому нужно отфильтровать колонки и оставить только те, где есть хоть один пропуск:   
```
missing = missing[missing > 0]
```
Разберемся с тем, как работает эта строка. Это называется **булевая индексация**. `missing` - это Series. Индексы = имена колонок, значения - кол-во пропусков. Вычисление идет изнутри наружу. Сначала вычисляется `missing > 0` (операции индексации вычисляются первыми) - возвращает тот же Series, но с булевыми значениями. То есть напротив наименования колонки у нас теперь не кол-во пропусков, а булева величина (True - если есть пропуски, то значит значение было больше нуля, соответственно True, если пропусков нет (то есть напртив наименования колонки 0), то получается False). Заметьте, что длина Series на этом этапе не меняется. Мы в скобках просто создали **булеву маску**.    
Далее идет вычисление `missing[...]`, мы применяем эту булеву маску к Series. Квадратные скобки с булевой серией внутри = фильтр. Остаются только те строки, где условие было True. То есть Series оставляет только те элементы, где маска равна True.   
После этого идет присваивание результата.   
Применение вот таких масок - это синтаксис Pandas. На чистом Python такое не получится сделать со списками. Но в чистом Python применяется другой подход - **генераторы списков** (с нимим мы познакомимся позже). 

В противовес фильтрации рассмотрим трансформацию, с котрой мы встретимся на этапе feature engineering. Возьмем вот такую строку для рассмотрения:  
```
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
```
Эта строка создает новый признак на основе имеющегося в датасете. Метод `astype()` рассматривать сейчас не будем, наша цель - разобрать то, как сравнение работает внутри квадратных скобок и снаружи. Как работает внтури мы разобрали, теперь посмотрим как работает снаружи.   
`train['Fireplaces']` - это объект Series (колонка), которая содержит в себе данные о количестве каминов в доме. И в коде выше мы проводим векторизированную операцию - поэлементно сравниваем значения с 0. И после сравнения мы получаем то же самый Series, то вместо числовых значений - True/False. А далее мы преобразовываем их в числа 0/1.   

Почему мы не можем применить булеву маску в этом случае? Например:   
```
train['HasFireplace'] = train[train['FirePlaces'] > 0]
```
Дело в том, что в этом случае мы бы собрали все колонки, где есть хотя бы один камин и присвоили бы это все новому признаку. Но получили бы ошибку, так как из Series с 1460 строками, мы бы получили Series с, например, 800 строками. И попыталсь бы присвоить итог Series с 1460 строками.   
Важно помнить, что при **фильтрации** мы отсеиваем ненужно, Series уменьшается в размерах, а при **трансформаци** мы приравниваем значения к True/False, но ничего не исключаем, Series имеет прежнюю длину.  

Также, дополнительно разберем метод `value_counts()`. Это один из самых полезных методов для работы с категориальными параметрами. Это метод объектов класса Series. Он работает следующим образом: подсчитывает количество уникальных значений в объекте Series и возвращает новую Series, где индекс - уникальные значения из исходной Series, а значения - количество раз, которое каждое значение встретилось. Есть много параметров, но наиболее полезным является `dropna`. При значении False, в Series попадает значение NaN, если же значение True, то NaN откидывается и мы получаем статистику только по реальным значениям.  

### Preprocessing

Теперь начнем заполнять пропуски. Пропуски заполняются с помощью метода `filna()`. Это метод объектов класса NDFrame. В метод передается значение, котором заполняется пропуск:   
```
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
```
С числовыми признаками также, только вместо строки - число.    

Также нужно помнить о том, что есть признаки, в которых пропуски нужно заполнять медианой. Например, в House Prices есть признак "LotFrontage". Он означает ширину участка по фасаду, то есть длину той стороны земленного участка, которая граничит с дорогой или улицей. Имеет числовой тип.    
В учебном датасете имеет множество пропусков. И просто заполнить медианой их нельзя. Для грамотного препроцессинга нужно заполнить пропуски медианой по районам (Neighborhood). То есть для одного района высчитывается одно медианное значение LotFrontage, для другого района - другая медиана.   

В этом поможет метод `groupby()`. Это метод определен в NDFrame. Он разбивает данные на группы по ключу (значению столбца или индекса), выполняет функцию для каждой группы независимо и собирает результат обратно в объект DataFrame или Series. Важно понимать, что сам по себе `groupby()` ничего не вычисляет, он лишь создает объект - "инструкцию", который хранит информацию в группах. Вычисления начинаются только после вызова метода агрегации (.sum(), .mean(), .transform() и т.д.). Это называется **ленивым вычислением**.   

Рассмотрим принцип работы `groupby()` более подробно на примере с признаком LotFrontage:  
```
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
```
В примере выше мы разбиваем весь датасет на группы по значениям признака Neighborhood. У нас есть 25 уникальных значений в колонке Neighborhood. А значит весь датасет DataFrame разбивается на 25 частей и превращается в DataFrameGroupBy. Во вторых скобках (квадратных) мы указываем еще один признак, со значениями которого мы будем работать. И на этом этапе мы получаем уже объект SeriesGroupBy, который содержит 25 групп колонок со значениями LotFrontage. И далее мы применяем метод `transform()`. Это метод объектов класса `GroupBy`, который является родительским для классов `DataFrameGroupBy` и `SeriesGroupBy`.   
Так вот, `transform()` принимает какой-то блок кода, который потом выполняется для каждой из 25 групп. В данном случае мы передаем в `transform()` lambda-функцию, в которой `x` - это группа (каждая из 25). И мы к каждой группе применяем метод `fillna()`, который принимает значение медианы каждой группы. `median()` - это метод объектов NDFrame. Он вычисляет медиану для каждой их 25 групп.   

Таким образом, мы вычисляем медиану на каждой группе и заполняем пропуски в каждой группе этими медианами.   

Если же так получилось, что у нас есть целлый район, где нет вообще никаких данных признака LotFrontage, то мы должны заполнить их глобальной медианой:  
```
if train['LotFrontage'].isnull().any():
    global_median = train['LotFrontage'].median()
    train['LotFrontage'] = train['LotFrontage'].fillna(global_median)
```
Метод `any()` - это метод класса NDFrame. Данный метод проверяет, есть ли хоть один элемент, который оценикается как True.    
И в коде выше, мы преобразовываем все ячейки LotFrontage в True/False и применяем метод `any()`, так как True будет являться пропуском, и если есть хотя бы одно True - оно будет заполнено глобальной медианой.  

Далее удалим строки, которые не нужны. Смотрим на количество пропусков по столбцам, и если видим, что в каком-то столбце есть 1-2 пропуска, то можно просто удалить строку, которая образует этот пропуск. На общую картину это не повлияет.   
Строки удаляются с помощью метода `dropna()`. Это метод объектов класса NDFrame. Данный метод удаляет метки (строки или столбцы), содержащие пропущенные значение (NaN или None). По умолчанию `dropna()` не изменяет объект, а возвращает новый, поэтому необходимо использовать параметр `inplace=True` или присваивание ноаой переменной. Если применяем к DataFrame, то важно использовать аргумент `subset='...'`, в котором указывается название колонки. В этой колонки проверяются пропуски и удаляются те строки, в которых припущены значения в указанной колонке:   
```
train.dropna(subset='Electrical', inplace=True)
```

После того, как мы заполнили все пропуски, необходимо устанить выбросы. Выбросы устраняются путем логарифмирования целевой переменной. Если у нас есть в выборке несколько очень дорогих домов, то это значит, что нужно сжать перечень цен, иначе это будет сбивать модель с толку.   
Логарифмирование - это принцип, при котором вместо обучения модели на абсолютных значениях, мы обучаем на логарифмах значений. Это позволяет сжать большие значения и растянуть маленькие. Будем использовать натуральный логарифм для этих целей. Это делается с помощью функции из NumPy `numpy.log1p()`. Эта функция - натуральный логарифм (логарифм по основанию $e$ (~2.718)), к аргументу которого прибавляется 1. Зачем прибавлять к аргументу единиицу? Это защита от нулевых значений. Даже есть значение вдруг будет нулевым, то мы прибавим к нему единицу и значение станет положительным.   

Каким образом логарифмирование помогает? Вот пример в контексте данного соревнования:  

| Цена ($) | `ln(Цена)` | Прирост логарифма | Комментарий |
|:---------|:-----------|:------------------|:------------|
| 10 000   | ~9.21      | —                 | Базовый уровень |
| 100 000  | ~11.51     | +2.30             | ×10 к цене → +2.3 к логу |
| 500 000  | ~13.12     | +1.61             | ×5 к цене → уже меньший прирост |
| 700 000  | ~13.46     | +0.34             | ×1.4 к цене → совсем маленький прирост |   

Как видно из таблицы, в абсолютных числах разрыв между 10 000 и 700 000 просто огромный. Но если взять натуральный логарфм от этих чисел, то разрыв между ними получится минимальным, чуть больше 4.   

Дело в том, что логарифм измеряет не абсолютное изменение, а относительное. Вот взять, к примеру, логарифм по основанию 10. Этот логарифм отвечает на вопрос: "В какую степень возвести основание 10, чтобы получить это число". Именно поэтому мы получаем такие числа. И это помгает более качественно обучить модель. А потом эти логарифмы можно вернуть в абсолютные значения.   

Теперь то, как это реализуется на практике:  
```
train['SalePrice_log'] = np.log1p(train['SalePrice'])
```

### Feature Engineering  
Теперь переходим к следующему этапу - создание новых признаков. В контексте данного соревнования можно создать следующие признаки:   
- Общая жилая площадь. В этом признаке мы просто складываем площадь подвала и площади этажей дома. Сложение колонок - векторизированно. Все как в линейной алгебре, сложение поэлементно. Только нужно учитывать то, что это сложение идет не по порядковому номеру строки, а по индексам. И если в какой-то колонке отсутствует индекс, то это вызовет ошибку при сложении и мы получим NaN на конкретном индексе, а не сумму числа и нуля. Поэтому важно перед feature engineering заполнить все пропуски, иначе модель просто упадет при встрече с ними. Вот так выглядит это сложение: `train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']`.   
- Возраст дома. Для добавления данного признака мы вычтем из года продажи год постройки. Такая же векторная операция, как и предыдущая: `train['HouseAge'] = train['YrSold'] - train['YearBuilt']`.  
- Время с последнего ремонта. Вычитаем из года продажи год последнего ремонта: `train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']`.   
- Взвешенное общее количество ванных комнат, приведенное к эквиваленту "полных ванн". В этом признаке мы складываем полноценную ванну над землей (ванна/душ + туалет + раковина), гостевой туалет над землей (только туалет + раковина), полноценная ванная в подвале и гостевой туалет в подвале. У полноценных ван коэффициент 1, а у неполноценных - 0.5. Эти коэффициенты - индустриальный стандарт в риелторской аналитике. В коде это выглядит вот так: `train['TotalBath'] = train['FullBath'] + 0.5*train['HalfBath'] + train['BsmtFullBath'] + 0.5*train['BsmtHalfBath']`.  
- Наличие гаража. Чтобы узнать это, мы используем площади гаража в домах и сравниваем их с нулем. Получаем True или False, а дальше переводим данные из булевой величины в 1/0. Это делается для того, чтобы отделить факт наличия от размера/количества. Это важно, так как дом с гаражом 15 м² и дом без гаража 0 м² сильно арзличаются по цене. Но для модели 0 и 15 лежат на одной числовой прямой. Модель может решить, что гараж 1 м² почти ничего не стоит, хотя сам факт наличия гаража повышает ликвидность. `train['HasGarage'] = (train['GarageArea'] > 0).astype(int)`.  
- Наличие подвала. То же самое, что и с гаражом.  
- Наличие камина. То же самое, что и с гаражом.   

Отдельно разберем метод `astype()`. Это метод объектов класса NDFrame. Возвращает объект того же типа, к которому он применялся. На вход принимает Python-тип (в данном случае int) и превращает данные из Series или DataFrame в этот тип.   

### Feature Transformation
Теперь перейдем к кодированию признаков. Поскольку модель не может работать с текстом, а только с числами. Нужно преобразовать все полученные данные в числа. Поэтому разделим все признаки на 3 типа: Numeric (числовые), Ordinal (порядковые) и Categorical (категориальные).   

**Numeric features** не нужно преобразовывать, так как они и так будут понятны модели. **Ordinal features** - это порядковые признаки. То есть если мы говорим о качестве камина, то он имеет несколько значений: Ex (Excellent, лучшее качество), Gd (Good, хорошоее качество), TA (Average, среднее качество) и так далее. Они должны быть расположены строго от худшего к лучшему. Это необходимо для правильного перевода в число. Все эти возможные значения будут представлены в массиве, и числовое значение будет присвоено в зависимости от индекса. То есть самое худшее будет иметь значение 0 (так как будет находиться на начальной позиции в массиве). **Categorical features** - это обычные строковые параметры. Например, параметр "фундамент" имеет возможные значения: "кирпич и черепица", "шлакоблок", "бетонный монолит" и другие. Тут не возможно сказать, что что-то лучше, а что-то хуже.     
Более подробно о том, как преобразуются в числа эти значения, будет дальше.   

Итак, для начала следует создать словарь для порядковых признаков. Ключами словаря будет наименование признака, а значениями будет массивы, где в правильной последовательности указаны все возможные значения этих признаков. Это необязательный пункт, но он нужен для удобства, так как из этого словаря мы будем формировать массивы для числовых параметров. И если потребуется что-то изменить, то проще будет внести изменение в словарь, из которого изменения перейдут в массивы, чем исправлять сначала один массив, а потом другой.    

In [ ]:
ordinal_mapping = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NoBasement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NoGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NoPool', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Fence': ['NoFence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

После этого можно перейти к формированию массивов с различными группами признаков.    

Сначала сформируем массив числовых признаков. Это делается с помощью метода `select_dtypes()`. Это метод объектов класса NDFrame. Он возвращает объект того же типа, к которому применялся. Суть метода заключается в выборе подмножества колонок. Имеет параметр `include`, который должен быть приравнен к какому-то типу. Например, если мы хотим получить колонки только числового типа, то аргумент должен выглядеть так: `include=['int64', 'float64']`. Если хотим только строковые данные, то так: `include=['object', 'str']`. Причем `object` уже является устаревшим, и его скоро уберут (вместо него нужно использовать str/string). При применении к DataFrame возвращает также DataFrame, но только с колонками того типа, который указан в include. Также у данного метода есть нюанс: в pandas/numpy bool считается подтипом целых чисел, это нужно учитывать. Поэтому при передаче значения "number" нужно быть осторожнее, если не нужны булевы значения, то лучше писать конкретные типы, такие как int64 и float64. И немного поговорим про то, как этот метод работает в Series. Поскольку данный метод вычленяет колонки необходимого типа, а Series в pandas - это однородный контейнер и, соответственно, все элементы в Series имеют один dtype, то логика на Series вырождается в простую проверку. Если dtype в Series совпадает с include - метод возвращает ту же самую серию целиком, если не совпадает, то возвращает пустую серию.   

После `select_dtypes()` у нас на руках DataFrame, но только с теми колонками, которые нам подходят по типу. А нам нужно только наименования этих колонок. Чтобы получить наименования, используется `columns`. Это атрибут-свойство из класса DataFrame. Он возвращает объект `pandas.Index` - это специальная упорядоченная структура данных pandas. Она достаточно сильно отличается от списка в Python и даже от списоков ndarray из NumPy. `pandas.Index` неизменяем, имеет метаданные и является строго однородным.    
Из-за его особенностей, он привязан к pandas-контексту, и при передаче его sklearn, json или API это может вызвать FutureWarning или скрытые баги. Также возникают проблемы с строгой типизацией и сериализацией.   

Поэтому мы будем использовать метод `to_list()`. Это метод из pandas, но оригинальное название пришло из NumPy и исторически `tolist()` был методом NumPy объектов `ndarray`. Данный метод принадлежит исключительно одномерным структурам: `pandas.Index`, `pandad.Series` и `numpy.ndarray`. Данный метод возвращает обычный Python-список.   

После этого у нас появляется массив - `numeric_features`, где собраны все числовые признаки (наименования). Но есть нюанс. В оригинальном train у нас есть такие числовые колонки, как Id, SalePrice и SalePrice_log. Они нам тут не нужны, так как мы формируем массивы, которые в дальнейшем будут использоваться для обучения структруы, которая будет преобразовывать DataFrame в понятные для модели матрицы. И на этих матрицах модель будет обучаться. А для обучения модели нужно убрать лишние столбцы (так как Id не имеет информационной ценности, а цены должны быть скрыты от модели).    

Для этого применим **List Comprehension** с фильтрацией. Это идиоматический паттерн Python для создания нового списка на основе существующего с отбором элементов по условию. Имеет следующую структуру: `[элемент for элемент in исходный_список if условие]`. На реальном примере это выглядит вот так: `numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]`.    

То есть в блоке `for col in numeric_features` мы перебираем исходный массив со всеми числовыми элементами. В блоке `if col not in ['Id', 'SalePrice', 'SalePrice_log']` ставим условие, что перебираемый элемент не должен быть в указанном массиве (в массиве указываем ненужные колонки). А `col`, который в самом начале - это то, что заносится в новый массив. Также не забываем, что весь этот алгоритм должен находиться в квадратных скобках.    

Теперь создадим массив из порядковых колонок. Для этого воспользуемся созданным ранее словарем и методом `keys()`. Это метод из встроенного класса `dict` в Python. То есть данный метод применяется к словрю для того, чтобы получить перечень ключей этого словаря. Метод возвращает объект типа `dict_keys`. Этот объект не является списком ключей, скорее "окном" в ключи словаря. Если добавить/удалить ключ из словаря, то объект dict_keys обновится мгновенно. Чтобы получить именно массив ключей, необходимо воспользоваться функцией `list()`.   

Функция `list()` определена в модуле `builtins`. Она принимает любой итерабельный объект (который можно перебрать в цикле) и создает новый изменяемый массив в памяти, последовательно извлекая элементы из итератора и записывая их в новый список. Возвращает объект типа `list`.    
Нельзя путать функцию `list()` с методом `to_list()`. Первое работает с любым итерируемым объектом и определено в Python, второе - является методом numpy/pandas и работает только на ndarray, Series и Index.   

В коде это выглядет так: `ordinal_features = list(ordinal_mapping.keys())`.   

Далее проделываем то же самое, но уже с категориальными признаками. В этом случае мы воспользуемся уже известными методами `select_dtypes()`, только отбирать нужно будет уже не числовые колонки, а строковые. В этом случае аргументу `include` нужно передавать значение `object`, но оно уже устарело, поэтому теперь нужно передать `string` ли `str`. Либо, чтобы наверняка, можно передать `['object', 'str']`.   

На основании получившегося массива создадим новый, так как строковые признаки могут быть как порядковыми, так и категориальными. А значит в текщем массиве у нас мешанина из строковых категориальных и порядковых признаков. Для решения пробемы воспользуемся **List Comprehension**. Условием будет следующим: `if col not in ordinal_features`. То есть в новый массив должны попасть все признаки, кроме тех, что есть уже в массиве ordinal_features. В коде это будет выглядеть так:  
```
categorical_features = train.select_dtypes(include=['str']).columns.to_list()
categorical_features = [col for col in categorical_features if col not in ordinal_features]
```

Также создадим еще один массив - `categories_list`. Это будет массив, в котором каждый элемент - это массив со значениями порядковых признаков, выстроенных в правильном порядке. Он потребуется нам позже. Создавать этот массив будет так же, как и предыдущие, с помощью **List Comprehension**: `categories_list = [ordinal_mapping[col] for col in ordinal_features]`.    

Теперь разберем причину, по которой нам нужен был словарь более подробно. Далее мы будем создавать энкодеры. Их будет всего 2 - энкодер для категориальных признаков и энкодер для порядковых признаков. 

Энкодер реализует замену строковых признаков на числа, чтобы математическая модель могла их "понять". Всего существует 2 вида энкодеров: OrdinalEncoder (порядковый) и OneHotEncoder (категориальный).  

**OrdinalEncoder** переводит порядковые признаки в числа. Ранее мы уже создавали словарь, в котором ключи = порядковые признаки, а значения = массив из возможных значений. Мы тогда расставляли их в порядке от худшего к лучшему. Это нужно, чтобы энкодер мог присвоить корректное число значению. Самый худший признак идет первым в массиве возможных значений. И поскольку он стоит первым, ему причисляется индекс 0. Следующее значение, которое чуть лучше, имеет уже индекс 1. И так далее. Таким образом, все возможные значения порядковых признаков переводятся в числа исходя из их позиции в массиве.   

**OneHotEncoder** переводит категориальные признаки в числа. Значения категориального признака не может быть хуже или лучше остальных. К примеру, если мы говорим о цвете забора (этого параметра нет в данных, просто пример), то он может иметь значение "желтый", "зеленый" или "красный". Мы не можем сказать, что какой-то цвет лучше, а какой-то хуже. OneHotEncoder кодирует такие признаки следующим образом: вместо одной колонки категориального признака формируется несколько колонок, количество которых зависит от количества возможных значений признака. Эти колонки бинарные и могут преобретать всего 2 значени - 0 или 1. Если забор имеет зеленый цвет, то в колонке зеленого цвета ставится 1, а остальные колонки, соответственно, преобретают значение 0. И так со всеми категориальными признаками.   

Очень важно не перепутать тип признака, иначе модель будет думать, что, например, зеленый цвет забора лучше красного. Или же, например, что лучшее качество камина ничем не отличается от среднего.    

Сами энкодеры ничего не преобразовывают. Это лишь "формы", которые содержат правила преобразования для каждой категории признаков. Приступим к реализации в коде. Для начала импортируем классы `OrdinalEncoder` и `OneHotEncoder` из пакета `sklearn`. Они находятся в подпакете/модуле `preprocessing`.    
А потом создадим 2 энкодера:       

```
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

ordinal_encoder = OrdinalEncoder(
    categories=categories_list,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)
```
В OrdinalEncoder мы передаем список из массивов, в которых все значения выстроены в правильном порядке (от худшего к лучшему). Также регулируем поведение энокодера в случае, если он на тестовых данных встретит незнакомое значение, которого не было в тренировочных данных: если встретит - пусть присовит этому значению -1.    
В OneHotEncoder также устанавливаем паттерн поведения при встрече с незнакомыми занчениями: энкодер просто будет игнорировать данное значение, и во всех бинарныз колонках признака проставит нули. А итоговой матрицой будет в формате numpy.ndarray.   

После реализации всех энкодеров нужно собрать их все вместе. Так как каждый энкодер отвечает за конкретные признаки, а реальный датасет состоит из множества разных признаков, нам потребуется инструмент, который сможет "связать" эти энкодеры вместе, чтобы их можно было применить к реальному датасету.   
Для этого необходимо создать объект `preprocessor` класса `ColumnTransformer`. Данный класс находится в модуле `sklearn.compose`. Импортируем его, а далее реализуем preprocessor:   
```
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers = [
        ('num', 'passthrough', numeric_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('cat', categorical_encoder, categorical_features)
    ],
    remainder = 'drop',
    sparse_threshold = 0.3,
    verbose_feature_names_out = True
)
```
В данном случае были использованы все параметры, которые только есть у класса ColumnTransformer. Всего их 4. Первый параметр `transformers`. Это единственный обязательный параметр класса. Он всегда должен присутствовать в инициализации объекта. Принимает массив кортежей, каждый кортеж состоит из произвольного наименования трансформера, энкодера и списка признаков. В числовом кортеже вместо энкодера используется строка 'passthrough', это означает, что числовые колонки нужно передать в выходную матрицу как есть, без каких либо изменений.    
Далее идет параметр `remainder`. В данном случае он имеет значение 'drop', что означает "исключить все колонки из X_train (передаваемого датасета), которых нет в numeric_features + ordinal_features + categorical_features".   
Параметр `sparse_threshold` имеет базовое значение в 0.3, его можно не трогать, а оставить по умолчанию. Это значит, что если в выходной матрице будет ненулевых значений меньше 30% - выходная матрица будет иметь формат scipy.sparse (разряженная матрица).   
Параметр `verbose_feature_names_out` имеет значение True, что означает, что к наименованиям колонок будет приписано "name__", где "name__" - это произвольное название трансформера. То есть колонка, например, "FireplaceQu" будет иметь в выходной преобразованной матрице название "ord__FireplaceQu".   

Также необходимо понимать, что порядок колонок в выходной матрице строго соответствуют порядку кортежей в transformers, затем идут колонки из remainder (если remainder='passthrough', а не 'drop', как в нашем случае). То есть в нашей выходной матрице будут иди колонки в следующем порядке: числовые -> порядковые -> категориальные. В конце "хвост" (но не в нашем случае, так как remainder='drop').   

Теперь, когда мы закончили с проектированием preprocessor-а, нужно разделить наш исходный датасет. У preprocessor-а, как и у любого другого трансформера (ColumnTransformer просто один из множества трансформеров в sklearn) есть 3 ключевых метода: `fit()`, `transform()` и `fit_transform()`. И как мы знаем, preprocessor нужен для преобразования исходного датасета в числовую матрицу. Так вот этим занимается метод `transform()`, но до этого метода нужно вызвать метод `fit()`, так как сначала мы обучаем preprocessor, а только потом он преобразовывает датасет в матрицу. И в понимании разницы между `fit()` и `transform()` кроется ключ к пониманию того, почему обучать preprocessor нужно на части данных, а не на всех сразу.   

Сначала разберемся, что такое метод `fit()`. Когда мы передаем датасет в метод `fit()`, preprocessor сканирует датасет, находит в нем уникальные категории, сортирует их, считает частоты и самое главное - строит словарь-правило и запоминает его внутри себя. Казалось бы, зачем все это нужно, если мы и так самостоятельно отсортировали необходимые колонки на числовые, порядковые и категориальные признаки, передали всю необходимую информацию в энкодеры и упаковали это все в preprocessor. Но на самом деле, preprocessor самостоятельно ничего не учит. При вызове `fit()` у preprocessor-а, он берет наш датасет (X_train), находит в нем колонки из ordinal_features и categorical_features. Проверяет, что они существуют (если нет, то кидает ValueError). Далее он вызывает метод `fit()` у каждого энкодера по отдельности: `ordinal_encoder.fit(X_train[ordinal_features])` и `categorical_encoder.fit(X_train[cat_features])` (когда передаем в квадратные скобки в DataFrame список, pandas интерпретирует это как "верни новый DataFrame, содержащий только те колонки, которые есть в переданном массиве"). Каждый энкодер пробегает по переданным колонкам и создает в памяти быструю хеш-таблицу. Например, ordinal_encoder выглядит так после `fit()`:
```
self.mapping_ = {
    'ExterQual': {'Po':0, 'Fa':1, 'TA':2, 'Gd':3, 'Ex':4},
    'BsmtQual': {'NoBasement':0, 'Po':1, ...}
}
```
А categorical_encoder выглядит как упорядоченный список numpy.ndarray.   
`fit()` запоминает, какие категории фактически встретились. Если в categories_list было ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], а в данных только ['TA', 'Gd'], энкодер все равно создаст маппинг для всех 6 значений, но будет знать, что остальные пока не встречались.   

Метод `transform()` же просто применяет все эти правила и преобразовывает датасет. Берет готовый словарь-правило (созданные энкодерами хеш-таблицу) и просто заменяет текст на числа.    

Теперь перейдем к ответу на вопрос: "Зачем разбивать датасет на части для обучения preprocessor-а?". Обычно разбивают на 2 части с соотношением 80/20. 80% - это часть, на которой будем тренировать preprocessor, а 20% - это валидационная выборка. Это делает для того, чтобы не случалось Data Leakage (утечка данных). Дело в том, что во время вызова метода `fit()` preprocessor сканирует датасет, находит уникальные категории, сортирует их и строит словарь-правило. Мы более подробно разобрали выше, этим всем занимаются по сути энкодеры. И сложность процедуры заключается в том, что правила замены формируются из данных. И если мы дадим preprocessor-у "посмотреть" на 100% данных, то случиться эта самая утечка данных, ведь он будет выстраивать данные преобразования с учетом будущего. Рассмотрим пример для лучшего понимания для OneHotEncoder с categories = 'auto':   

Допустим, в train (80%) район Neighborhood имеет 20 уникальных значений. В val (20%) случайно попал 21-й район: 'NewDistrict'. Neighborhood - это категориальный признак. В нашем случае, у OneHotEncoder параметр handle_unknown равен 'ignore', соответстввенно, все новые значения, которые энкодер увидит в тестовом датасете - будут проигнорированы, а имеющиеся колонки получат по нулям. Когда мы передаем в preprocessor данные, мы передаем лишь энкодер и массив наименований категориальных колонок. То есть энкодер (категориальный) знает, какие конкретно колонки он должен трансформировать, но всех возможных значений он не знает. И во время обучения он строит ограниченое количество колонок, только под те значения, которые встречает. И если мы обучим его на 80%, то на валидационной выборке он встретит 21-ое значение, и будет вынужден проигнорировать его, а во все 20 колонок проставить нули. Если бы мы обучали энкодер на 100% данных, то энкодер заранее знал бы, сколько всего колонко нужно построить, и построил бы 21 колонку, в однй из которых стояла бы единица. Из-за вот таких различий модель может давать завышенный результат на валидационной выборке, если preprocessor обучался на 100% данных.    

Таким образом, нам необходимо тренировать preprocessor на 80% тренировочных данных, чтобы сразу видеть, какой результат способна выдавать модель. А не удивляться, почему на тренировочных данных хороший результат, а на тестовых - нет. 

### Split

### Modeling  

## Функции, атрибуты и методы Pandas   
`pandas.read_csv(path)` - функция из pandas, принимающая на вход путь к csv-файлу, и преобразующая файл в объект DataFrame.   
`info()` - метод объектов NDFrame. Возвращает "None". Показывает количество столбцов и строк в датасете, а также показывает количество непустых значений в каждом столбце. Чаще всего применяется к объектам DataFrame. Работает как функция "print()", просто выводит данные в консоль. Сохранить их в переменную не получится, так как возвращается None.       
`describe()` - метод объектов NDFrame, возвращает тот же тип объекта, для которого был вызван. Показывают различную статистику по данным: минимум, максимум, среднее и т.д. Активно применяется как для объектов DataFrame, так и для объектов Series. При применении метода к Series получаем полную статистику по колонке. При применении к DataFrame, получаем таблицу, где строки - это параметры (mean, std, max,...), а столбцы это изначальные столбцы из DataFrame. По умолчанию считает статистику только для числовых данных, но если нужна статистика по строчным, то используется параметр "include" со значениями "all/str(string)". В случае применения include='all' - отображает данные по всем колонкам (и числовым и строковым), только для строковых одни данные отображаются (unique, top, freq), а для числовых колонок другие (mean, std, ...). Исходный объект не меняет, но результат можно присвоить переменной. Аргумента "inplace" у метода нет.    
`isnull()` - метод объектов NDFrame. Возвращает объект того же типа, к которому применяется, только заполненный "True/False" (True - есть пропуск, False - нет пропуска). Нет параметров. Исходный объект не меняется, метода "inplace" нет.   
`sum()` - метод объектов NDFrame. Суммирует значения. Если применяется к DataFrame - возвращает Series. Если применяется к Series - возвращает scalar (одно число). Работа метода с Series понятна. В случае работы с DataFrame играет значение параметр "axis". Если он равен "0", то сложение идет по вертикали и возвращается Series, где индексы - это наименования колонок DataFrame, а значения - сумма значений каждой колонки DataFrame. Если же "axis" равен "1", то сложение идет по горизонтали. И мы получаем Series, где индексы равны индексам DataFrame, а значение - это сумма всех значений строки. Также важно понимать, что в современных версиях pandas есть еще параметр "numeric_only", который по умолчанию False. То есть в сложении будут участвовать все типы, и числовые и строковые (конкатенация). Но если в одной строке или столбце встретятся и строки и числа, то Pandas выдаст ошибку TypeError. При значении True, все нечисловые колонки будут проигнорированы. Исходный объект не меняется, параметра "inplace" нет. Имеет аналогичную функцию в Python, однако принцип работы разный.   
`sort_values()` - метод, определенный отдельно для Series и DataFrame. Возвращает тот тип, к которому был применен. Причина, по которой он определен не в NDFrame, а по отдельности заключается в разных подходах к сортировке. При применении к Series все понятно, тут можно использовать параметр "ascending". Значение False означает сортировку по убыванию. Значение True означает сортировку по возрастанию. При сортировке DataFrame обязательно использовать параметр "by" (by='SalePrice' или by=['SalePrice']). Этому параметру нужно предоставить имя колонки, по которой будет осуществляться сортировка. Также можно сортиовать по нескольким столбцам. Для этого параметру "by" нужно передать список из названий столбцов (by=['Neighborhood', 'LotFrontage']). Он отсортирует их сначала по первому столбцу из списка. А потом, если в первом столбце были одинаковые значения, метод начнет сортировать по второму столбцу. Если все значения из первого столбца были разные, то второй сортировки не будет. Также важно помнить про аргумент "inplace", так как данный метод не вносит изменения в оригинал по умолчанию, а создает копию. Но на практике принято использовать присваивание новой переменной, нежели вносить изменения в исходный объект. Такой подход экономит память.      
`value_counts()` - метод объектов Series. Он применяется к Series, подсчитывает количество каждого уникального значения и возвращает Series, где индекс - это уникальное значение, а значение - это число, сколько раз встретилось какое-то знчение в изначальном Series. Есть аргумент dropna. Если оно False, то к уникальным значениям добавляется NaN (пропуски), если значение True, то пропуски игнорирются и не попадают в статистику. Метода inplace нет.    
`fillna()` - метод объектов NDFrame. Возвращает тот же объект, к которому применяется. Используется для заполнения пропусков в датасете. Аргументом является значение, которое мы хотим вставить на место пропуска. В случае применения к Series, все работает легко, просто заполняются пропуски в колонке. При применении к DataFrame все немного сложнее. Можно заменить все пропуски в DataFrame одним значением. Можно применить словарь для того, чтобы заменить пропуски разными значениями в зависимости от колонки (df.fillna({'A': 0, 'B': 99})). Тут мы заполняем пропуски в колонке 'A' нулями, а пропуски в колонке 'B' значениями 99. Данный метод относится к методам-трансформерам, поэтому в нем есть параметр inplace, но его лучше не использовать, так как данный параметр не экономит память.    
`groupby()` - это метод объектов NDFrame. Он применяется для того, чтобы сгруппировать данные. В первых скобках () указывается колонка, по которой нужно группировать. После этого возвращается объект DataFrameGroupBy, это тот же датасет, только разделенный на группы по уникальным значениям выбранной колонки. Далее во вторых скобках [] указывается вторая колонка, с чьими значениями мы собираемся взаимодействовать. После этого возвращается уже объект SeriesGroupBy. Это тот же самый Series, только поделенный на несколько групп. Далее мы можем применять какие-то функции, которые будут взаимодействовать с каждой группой. И важно понимать, что мы можем использовать на этих группах методы NDFrame, потому что каждая такая группа в SeriesGroupBy - это объект Series. Это было описание того, как он применяется к DataFrame. В случае с Series все тяжелее. Там сортировка может быть осуществленна по внешнему списку, либо по индексу, либо через словарь или функцию. На данный момент углубляться не будем.    
`transform()` - это метод объектов GroupBy. GroupBy - родительский класс для классов DataFrameGroupBy и SeriesGroupBy. Он принимает какой-то код, который применяется к каждой группе в объекте SeriesGroupBy или DataFrameGroupBy.    
`median()` - это метод объектов класс NDFrame. Он высчитывает 50% процентиль (медиану). В случае применения к Series возвращает scalar - медиану всех значений в колонке. В случае применения к DataFrame возвращает Series, где индекс - это названия столбцов (при axis=0) или индексы строк (при axis=1), а значения - вычисленные медианы.   
`any()` - метод объектов класса NDFrame. Проверяет, есть ли у объекта хотя бы один элемент, который оценивается как True. Если применить данный метод после "isnull()", то при применении к Series возвращает scalar (True (если есть пропуски) или False (если пропусков нет)). При применении к DataFrame возвращает Series (по одному True/False на каждый столбец).    
`dropna()` - метод объектов класса NDFrame. Имеет множество параметров. Если используется для DataFrame, то важно применять аргумент "subset='...'", в котором указывается наименование колонки, в которой проверяются пропуски. И удаляются все строки, которые имют пропуски в этой колонке. Также важно значть, что данный метод не совершает изменения в самом объекте, а возвращает новый по умолчанию. Поэтому важно использовать либо присваивание, либо аргумент "inplace=True".   
`astype()` - метод объектов класса NDFrame. Возвращает тот же тип объекта, к которому и был применен. Применяется к Series или DataFrame и преобразовывает данные в ячейках в тот тип, который был передан в метод.    
`select_dtypes()` - метод объектов класса NDFrame. Возвращает объект того же типа, к которому применяется. В качестве параметра используется "include", который принимает типы данных. Таким образом, данный метод вычленяет из данных только те колонки, которые соответствуют указанному типу в "include" (может принимаеть "number", который включает все числовые типы, а также bool; также принимает str/string/object, object устарел). На Series работает несколько иначе, чем на DataFrame. Поскольку Series - это однородный контейнер, где все значения одного типа, то при использовании "select_dtypes()" на Series, метод просто проверяет тип данных, и если они совпадают с тем, что указано в include, то возвращается исходный Series, а если нет, то пустой Series.    
`columns` - атрибует свойство объектов класса DataFrame. Возвращает "pandas.Index" - это неизменяемая система координат с хеш-поиском и выравниваением. При примененении к DataFrame будет содержать в себе перечень всех наименований колонок.   
`to_list()` - метод одномерных объектов (pandas.Index, pandas.Series, numpy.ndarray). Он не наследуется ни от какого класса, это независимая реализация в ndarray, Index и Series. Возвращает обычный Python list.   


## Функции и методы NumPy  
`numpy.log1p()` - функция из NumPy. Вычисляет натуральный лагорифм от аргумента + 1.   
`numpy.nan` - 


## Функции и методы Python  
`keys()` - метод класса dict из Python. То есть применяется к обычным словарям и возвращает объект "dict_keys", который не является списком, а скорее "окном" в ключи словаря. "dict_keys" - итерируемый объект.   
`list()` - это функция из Python. Принимает любой итерабельный объект и извлекает их него элементы, записывая в новый изменяемый массив.    


## Функции, классы и методы sklearn  
#### Подмодуль base
`TransformerMixin` - это базовый миксин (mixin), который предоставляет метод fit_transform() любому классу, реализующему fit() и transform(). Данный класс объединяет все трансформеры в sklearn, независимо от подмодуля. Класс является трансформером, если реализует методы fit() и transform(). В sklearn есть множество различных трансформеров, например, OrdinalEncoder, OneHotEncode, ColumnTransformer и многие другие. Во всех них определены методы fit() и transform() по отдельности, а fit_transform() уже идет "в наследство" от их общего родительского класса TransformerMixin.   

#### Подмодуль preprocessing       
`OrdinalEncoder` - класс из подмодуля preprocessing модуля sklearn. Наследуется от базового класса "TransformerMixin". Имеет несколько парметров:  
- *categories* - данный параметр принимает список значений в определенном порядке для порядковых признаков. Имеет 2 возможных значения: 'auto' или list[list]. Значение по умолчанию - 'auto'. Всегда нужно передавать свой собственный список, так как при 'auto' сортировка значений происходит в алфавитном порядке.   
- *drop* - удаляет лишнюю колонку при преобразовании строк в числа, решая тем самым проблему мультиколлинеарности. Мультиколлинеарность - это проблема, при которой наличие всех колонок в категориальном признаке мешает модели правильно определить вес параметра. Например, у нас есть категориальный признак, он имеет 3 возможных значения. Когда на одном значении 1, то на других 0. Нужно ли нам в этом случае 3 колонки, чтобы определить, какое значение имеет признак? Нет, достаточно двух, так как если в оставшихся двух колонках 0, значит 1 в третьем столбце. Почему мультиколлинеарность является проблемой? Потому что линейная модель имеет формулу "$Цена = w_1*x_1 + w_2*x_2 + w_3*x_3 + b$", и когда модели нужно найти одни единственные веса для $w_1, w_2, w_3$, возникает проблема с бесконечным количеством решений системы уравнений из-за связи $x_1 = 1 - x_2 - x_3$. Такая система может иметь следующие значени: "$w_1=100, w_2=200, w_3=300$", "$w_1=1000, w_2=700, w_3=300$", "$w_1=-500, w_2=800, w_3=300$". Все 3 варианта дают одинаковую ошибку на обучающих данных, компьютер не понимает какое значение выбрать, веса начинают "скакать", становятся огромными или отрицательными, модель становится нестабильной. Проблемой мультиколлинеарности страдают только линейные модели и только категориальные признаки (так как только они заменяются несколькими бинарными колонками, порядковые признаки такими проблемами не страдают). Данный параметр имеет несколько возможных значений: *None* - значение по умолчанию (никакого эффекта не оказывает, так как ни одна колонка не удаляется); *'first'* - удаляет первую категорию (превращает "0, 1, 2" в "0, 1"); *'if_binary'* - удаляет колонку только если у признака ровно 2 уникальных значения. Если >2, то оставляет все. В контексте OrdinalEncoder данный признак избыточен, его не нужно использовать вовсе.   
- *handle_unknown* - регулирует рекцию энкодера на незнакомое значение. Мы передаем в OrdinalEncoder массив из массивов возможных значений. Потом обучаем этот энкодер на этих значениях. И если в обучающей выборке какое-то значение не попадалось, но попалось в тестовой выборке, тогда энкодер ведет себя в соответствии с переданным значением в handle_unknown. Может принимать 2 значения (только для OrdinalEncoder, у OneHotEncoder всего 3 значения): 'error' и 'use_encoded_value'. Значение 'error' является значением по умолчанию. Означает, что если энкодер встретил незнакомое значение в тестовой выборке, значит нужно вернуть ошибку. Пайплайн упадет. Значение 'use_encoded_value' говорит, что при встрече незнакомого значения нужно присвоить ему вес, указанный в "unknown_value" - это еще один параметр класса OrdinalEncoder.   
- *unknown_value* - данный параметр отвечает за значение, которое будет присвоено значению, которое энкодер увидит впервые; работает только если *handle_unknown='use_encoded_value'*. К примеру, у нас есть параметр "качество камина". Есть перечень возможных значений, расположенных от худшего к лучшему = ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], а в тестовой выборке может попасться значение 'Super'. Энкодер не знает, насколько хорошее или плохое это значение относительно остальных, поэтому ему будет причислено число, которое не было занято другими значениями, и как правило, это -1. Параметр может принимать несколько значений: *int* - любое целое число (например -1); *float* - любое десятичное число; *np.nan* - NaN из NumPy. По умолчанию параметр имеет значение -2, но как говорилось ранее, обычно причисляют значение, предшествующее худшему, то есть -1.   
- *encoded_missing_value* - параметр, отвечающий за то, какое значение будет иметь пропуск в признаке. Снова рассмотрим пример с тем же признаком "качество камина". Изначально нам в руки попадает необработанный датасет, где есть много пропусков. Пропуски в датасете означают отсутствие камина вовсе. В "data_description.txt" четко написано, что NA - это отсутствие камина. Но из-за особенностей работы pandas, это "NA" превращается в NaN, что классифицируется как пропуск. Параметр encoded_missing_value позволяет обрабатывать все пропуски. К примеру, мы можем не обрабатывать пропуски в датасете, а оставить все, как есть. В этом случае даже не потребуется обрабатывать пропуски в учебном датасете, они просто будут заменены значением, которое мы присвоим encoded_missing_value. Данный пораметр может преобретать следующие значения: *int* - любое целое число; *foat* - любое десятичное число; *np.nan* - NaN из NumPy. По умолчанию этот параметр имеет значение NaN. Чтобы данный параметр работал, нужно указывать *handle_unknown='use_encoded_value'*.  

Есть важный нюанс, который следует учитывать. *handle_unknown + encoded_missing_value* - это параметр, отвечающий за пропуски в тренировочных данных. Если мы их не заполнили, то за нас это сделает "encoded_missing_value". И нам не нужно будет указывать NaN в массиве categories_list на первой позиции. Это за нас сделает энкодер. *handle_unknown + unknown_value* - рабатает только с НЕИЗВЕСТНЫМИ значениями в ТЕСТОВЫХ датасетах.   

Это значит, что при незаполненных пропусках в тренировочном датасете есть несколько вариантов развития событий: 
- Указываем handle_unknown='use_encoded_value'; unknown_value=-1; encoded_missing_value=-2. В этом случае, все пропуски принимают значения -2, а реальные значения начинаются с 0. Неизвестные значения в тестовых датасетах получают значение -1.  
- Указываем только handle_unknown='use_encoded_value'; unknown_value=-1. В этом случае получаем ошибку, так как модель не понимает, как интерпритировать пропуски. Пропуски не подходят под unknown_value, так как этот параметр предназначен только для неизвестных значений в тестовом датасете. Но это только для линейных моделей. Бустинг-модели нативно поддерживают np.nan и автоматически строят по ним оптимальные ветки сплита.  
- Указываем только handle_unknown='use_encoded_value' и encoded_missing_value=-2. В этом случае ошибки не будет, пайплан не упадет, но при встрече неизвестных значений в тестовом датасете, они будут получать значение -2, так как чтобы работал encoded_missing_value, нам нужно указать handle_unknown='use_encoded_value', и соответственно unknown_value тоже будет работать, но будет преобретать значение по умолчанию -2. И NaN будет равен тому же значению, что и любое новое значение в тестовом датасете.  
- Ничего не указываем = handle_unknown='error'. Пайплайн упадет.   

`OneHotEncoder` - класс из подмодуля preprocessing модуля sklearn. Наследуется от базового класса "TransformerMixin". Имеет несколько парметров:  
- *categories* - данный параметр принимает перечень всех значений категориальных признаков. Может принимать 2 значения: list[list] и 'auto'. В случае выбора list[list] нужно будет передать массив, где каждый элемент является массивом значений одного из категориальных признаков. В случае же выбора 'auto', энкодер сам совершает следующие действия: (1) Фильтрует (берет только валидные значения), то есть находит все уникальные значения для каждой категориальной колонки, уникальные значению ищутся раздельно, энкодер обрабатывает каждую колонку из categirical_features независимо друг от друга. Данный процесс начинает только тогда, когда энкодер попадает в preprocessor, куда мы передаем этот энкодер и массив категориальных признаков. (2) Сортирует (все уникальные значения, которые он признал валидными); Порядок самих категориальных колонок энкодер не трогает, он следует строго по списку categorical_features, а вот все значения для кадой колонки он сортирует в алфавитном порядке (если строки) или по возрастанию (если числа). (3) Запониминает отсортированный список в атрибут "categories_[i]". categories_[i] - это список массивов (list of ndarrays). i - это индекс элемента массива categorical_features. То есть мы можем использывать запись categorical_encoder.categories_[0] и увидеть отсортированный список всех уникальных значений для первой категориальной колонки. categories_[i] появляется только после применения метода fit() у preprocessor-а. (4) Создание матрицы; во время применения метода transform() у preprocessor-а, энкодер создает матрицу из множества бинарных колонок. Количество колонко этой матрицы = суммарное количество элементов в атрибует "categories_" (но только, если параметр drop=None, и соответственно, он не удаляет никакие колонки).  
- *drop* - данный параметр удаляет лишнюю колонку при преобразовании строк в числа, решая тем самым проблему мультиколлинеарности. Понятие мультиколлинеарности более подробно разобрано в этом же параметре в OrdinalEncoder. Только стоит напомнить, что проблемой мультиколлинеарности страдают только линенйые модели в категориальных признаках. В OrdinalEncoder данный параметр был избыточен и мог принимать только 3 значения (None, 'first', 'if_binary'). В OneHotEncoder данный парметр может принимать 4 разных значения: *None* - отключает данный параметр, *'first'* - удаляет одну колонку в преобразованной матрице у каждого категориального признака. *'if_binary'* - удаляет колонку в преобразованной матрице у каждого категориального признака, но только в том случае, если в колонке всего 2 значения. Если признак имеет >2 значений, то оставляет все как есть. *array-like* - это список/массив длины N, где N - количество категориальных признаков. Каждый элемент списка указывает, какую именно категорию удалить для соответствующего признака. Если удалять не нужно - ставится None. Синтаксис выглядит так: drop=['Control', None, 'Red']. Это значит, что мы берем первый признак из categorical_feature и удаляем из преобразованной матрицы колонку 'Control', далее доходим до второго признака и ничего не трогаем, переходим к третьему признаку и удаляем из преобразованной матрицы колонку 'Red'. Зачем это нужно, если есть 'first'? Дело в том, что 'first' удаляет категорию, которая стоит первой после сортировки. Но в статистике и бизнес-логике часто нужно удалить конкретную смысловую категорию. array-like даёт полный контроль над тем, что станет референсной (базовой) группой в линейной модели.  
- *sparse_output* - параметр, отвечающий за формат данных, которые возвращает энкодер после преобразования. Формат данных может быть представлен в виде "сжатой" (разряженной) матрицы или обычного "плотного" массива. Может иметь значения: True или False (по умолчанию). При True энкодер возвращает разреженную матрицу (scipy.sparse). Она хранит только индексы и значения всех ненулевых элементов. Если у нас много категорий (сотни или тысячи), то это сэкономит много огромный объем оперативной памяти. При False возвращается обычный плотный массив (numpy.ndarrya). В памяти хранятся все нули и единицы. Нужно использовать, когда категорий мало или нужно сразу передать результат в библиотеку, которая не поддерживает разреженные форматы.  
- *dtype* - данный параметр отвечает за тип данных, в котором будут представлены числа в результирующей матрице. Этот параметр принимает любой стандартный числовой тип данных из библиотеки NumPy. По умолчанию имеет значение "numpy.float64". Но можно использовать "numpy.float32", если данных очень много. Это сокращает потребление памяти в 2 раза по сравнению с float64. Также можно использовать и целые типы, например, "numpy.int32" или "numpy.int8", если мы хотим видеть в матрице простые 0 и 1 без точек. Но проблема заключается в том, что в этом случае sklearn выдаст DataConversionWarning или сам скастит обратно во float, так как после энкодеров часто идут матричные операции, линейные модели или StandardScaler/MinMaxScaler, которые требуют вещественные числа. Поэтому для OneHotEncoder лучше оставлять float64/float32.   
- *handle_unknown* - данный параметр отвечает за то, как себя будет вести энкодер при встрече с незнакомым значением на тестовой выборке. В отличии от OrdinalEncoder, где данный параметр может приобрести только 2 значения, в OneHotEncoder данный параметр обладает 3 возможными значениями: *'error'* (значение по умолчанию, но только если не заданы min_frequency или max_categories) - выдает ошибку при встрече с незнакомым значением; *'ignore'* - игнорирует новое значение из тестовой выборки, выставяляя во все бинарные колонки признака нули; *'infrequent_if_exist'* - это значение, созданное для борьбы с high-cardinality (признаками с десятками/сотнями редких категорий). Это работает следующим образом: мы используем еще 2 параметра для обозначения порога редкости - *min_frequency=10* или *max_categories=20* (они взаимоисключающие, можно использовать только один из них). Во время вызова метода fit() у preprocessor-а энкодер считает частоту каждой категории (сколько раз встречается каждое уникальное значение в каждом признаке). Категории, встречающиеся реже порога, не получают отдельных колонок. Вместо этого они объединяются в одну общую колонку с именем {FeatureName}_infrequent_sklearn. При вызове метода transform у preprocessor-а, любая неизвестная категория из тестовой выборки попадает в эту же колонку. К примеру, если у нас есть признак Neighborhood, который содержит в себе 25 уникальных значений, и 20 из них встречаются реже 5 раз (то есть min_frequency=5), то после вызова метода fit() у preprocessor-а энкодер создаст 5 колонок для частых районов (те, которые встречаются чаще 5 раз) и одну колонку 'Neighborhood_infrequent_sklearn', куда будут определены все редкие районы.   
- *min_frecuency* - дополнительный параметр, который идет в связке с handle_unknown=infrequent_if_exist. Принимает: *int* (абсолютное количество значений), *float* (доля от общего числа значений, например: 0.01 означает, что признаки, встречающиеся реже, чем в 1% строк уйдет в общую колонку), *None*. Если какое-то значение признака встречается реже, чем указано в min_frequency, то оно лишается отдельной колонки и помещается в колонку {FeatureName}_infrequent_sklearn, где располагаются другие такие же редкие значенния, а также незнакомые значение из тестовой выборки.  
- *max_categories* - дополнительный параметр, который идет в связке с handle_unknown=infrequent_if_exist. Принимает: *int*, *None*. Данный параметр выбирает самые часто встречающиеся категории (топ N, где N - это значение данного парамтера) и оставляет их как отдельные столбцы. Все остальные (менее частые) категории объединяет в один общий столбец, который по умолчанию называется infrequent_category.  То есть если max_categories = 20, то энкодер выделит 19 столбцов под самые частые уникальные значения, а 20-й столбец станет "сборной" для самых редких значений.   

#### Подмодуль compose  
`ColumnTransformer` - класс из подмодуля compose модуля sklearn. Имеет множественное наследование, его основными родительскими классами являются TransformerMixin и BaseEstimator. Имеет несколько параметров:   
- *transformers* - данный параметр является обязательным и должен принимать данные типа *list[tuple(name, transformer, columns)]*. **name** означает наименование трансформера, может быть произвольной строкой; **transformer** - это энкодер, который должен быть создан до момента объявления объекта ColumnTransformer; **columns** - это список наименований колонок, массив с порядковыми, категориальными или числовыми колонками колонками. Как правило в параметр transformers попадает 3 кортежа, по одному на каждый тип признаков: числовой, категориальный и порядковый.   
- *remainder* - управляет судьбой колонок, которые физически присутствуют в исследуемом DataFrame, но не указаны ни в одном из массивов признаков (numeric_features, ordinal_features, categorical_features) внутри параметра *transformers*. Preprocessor анализирует те данные, что мы ему передали (X_train) и сравнивает их с объединененным списком numeric + ordinal + categorical. Если находит лишние колонки в X_train (те, которые не указаны в numeric + ordinal + categorical, например 'Id', 'SalePrice' или 'SalePrice_log', которые мы не помещали в numeric_features), то применяет правило remainder. Может принимать 3 значения: *'drop'* (значение по умолчанию) - удаляет из выходной матрицы те колонки, которые не обнаружены в numeric + ordinal + categorical; *passthrough* - оставляет как есть, добавляет в конец выходной матрицы в исходном порядке; *Transformer (объект)* - применяет этот трансформер ко всем "хвостовым" колонкам. В sklearn transformer - это архитектурный интерфейс. Любой объект, у которого есть методы .fit() и .transform() считается трансформером. И когда параметру remainder приравнивается *Transformer* - это означает, что все колонки, которые не перечислены в numeric + ordinal + categorical нужно обработать вот этим самым трансформером, вместо того, чтобы удалить или оставить сырыми. Например, трансформером может быть StandardScaler() - он автоматически масштабирует все неуказанные числовые колонки. Вообще, различных трансформеров в sklearn десятки.   
- *sparse_threshold* - это автоматический переключатель на выходе ColumnTransformer. Данный параметр отвечает за то, в каком виде будет выходная матрица - numpy.ndarray или scipy.sparse. Принимает в качестве значения исключительно float в диапозоне [0.0, 1.0]. По умолчанию имеет значение "0.3". При "0.0" всегда возвращает плотную numpy.ndarray. При "0.3" если доля ненулевых элементов (density) в итоговой матрице < 0.3 (30%) - возвращает scipy.sparse, иначе - ndarray. При "1.0" всегда возвращает разреженную матрицу (если хоть один трансформер выдал sparse). sparse_threshold не конфликтует с sparse_output из OneHotEncoder. Это пост-обработчик склейки, который автоматически переключает формат финальной матрицы при высокой разреженности. Данный парметр нужен в большенстве своем для экономии памяти. В NLP или рекомендательных системах матрицы часто на 99% состоят из нулей. Без sparse_threshold мы бы получил OOM (Out Of Memory) на склейке. Значение 0.3 является золотым стандартом и его редко когда нужно менять.  
- *verbose_feature_names_out* - данный параметр отвечате за добавление префикса "name__" к именам колонок. Принимает в качестве значений: True (по умолчанию) или False. При True это выглидит так: num__TotalSF, ord__ExterQual, cat__Neighborhood_CollgCr. "name" - это наименование трансформера из параметра transformers: num - numeric, cat - categorical, ord - ordinal.    
